
# V32 LIVE — Cascade + OOF Super-Ensemble

**Public-LB champion: V14 = 0.586**

This notebook is built ONLY for the live Kaggle ID set that already accepted V14/V25.

## Verified live partition

- Train: **3,297**
- Validation: **500**
- Test: **1,000**
- First test IDs: `BICH_0003, BICH_0006, BICH_0015, BICH_0016`
- Test IDs exactly match accepted V14.

No V28/V29/V30/V31 alternate-partition prediction is used.

## Architecture

### New models
1. Stage-1 binary BanglaBERT for `is_toxic`
2. Stage-2 attention-pooling BanglaBERT trained on:
   - every toxic training comment
   - hardest non-toxic comments according to Stage-1 OOF
3. Character + word TF-IDF OOF expert

### Reused LIVE OOF models
- V15 rare-label specialist
- V17 BanglaBERT toxic
- V17 XLM-R toxic
- V18 MuRIL
- V19 mDeBERTa toxic
- V23 conditional attribute model
- V25 BanglaBERT Large toxic
- V14/V2 validation + test probability family

### Clean stacking
A regularized meta-classifier is trained on **3297 TRAIN OOF rows only**.
The 500-row validation set is not used to fit the meta-classifier.

Validation is used only for final source/calibration selection, which the competition permits.

Send `v32_results.zip` to ChatGPT before submission.


In [ ]:

# 1. Upload V32 LIVE data
from google.colab import files
import pathlib, zipfile

uploaded=files.upload()
assert "v32_LIVE_data.zip" in uploaded

DATA_DIR=pathlib.Path("/content/v32_data")
DATA_DIR.mkdir(exist_ok=True)

with zipfile.ZipFile("v32_LIVE_data.zip") as z:
    z.extractall(DATA_DIR)

print("Extracted:")
for p in sorted(DATA_DIR.iterdir()):
    print(" ",p.name,f"{p.stat().st_size:,} bytes")


In [ ]:

# 2. Dependencies + BanglaBERT
import sys,subprocess

subprocess.check_call([
    sys.executable,"-m","pip","install","-q",
    "transformers==4.57.6","tokenizers==0.22.1","sentencepiece==0.2.1",
    "scikit-learn","scipy","pandas","numpy","tqdm","safetensors"
])

subprocess.check_call([
    sys.executable,"-m","pip","install","-q",
    "git+https://github.com/csebuetnlp/normalizer"
])

from transformers import AutoConfig,AutoTokenizer
MODEL_NAME="csebuetnlp/banglabert"
_=AutoConfig.from_pretrained(MODEL_NAME)
_=AutoTokenizer.from_pretrained(MODEL_NAME)

print("✅ BanglaBERT ready")


In [ ]:

# 3. Imports + STRICT LIVE-ID verification
import gc,re,math,json,random,unicodedata,zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from tqdm.auto import tqdm
from scipy.sparse import hstack

from transformers import (
    AutoTokenizer,AutoModel,AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from normalizer import normalize

SEED=2026
N_FOLDS=5
MODEL_NAME="csebuetnlp/banglabert"
EPOCHS=3
SNAPSHOT_EPOCHS=[2,3]
MAX_LENGTH=128
BATCH=8
ACCUM=2
EVAL_BATCH=32
LR=1.5e-5
HEAD_LR=3e-5
WEIGHT_DECAY=0.01
WARMUP_RATIO=0.10

LABELS=[
    "is_toxic","implicit_toxicity","sarcasm","contains_profanity",
    "threatening","identity_attack","sexual_content"
]
LI={l:i for i,l in enumerate(LABELS)}
ATTRS=LABELS[1:]

RUN_DIR=Path("/content/v32_output")
RUN_DIR.mkdir(exist_ok=True)

def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic=True
    torch.backends.cudnn.benchmark=False

seed_all(SEED)
assert torch.cuda.is_available(),"Enable T4 GPU."

train=pd.read_csv(DATA_DIR/"train.csv",dtype={"ID":str})
valid=pd.read_csv(DATA_DIR/"validation.csv",dtype={"ID":str})
test=pd.read_csv(DATA_DIR/"test.csv",dtype={"ID":str})
v2v=pd.read_csv(DATA_DIR/"v2_valid.csv",dtype={"ID":str})
v2t=pd.read_csv(DATA_DIR/"v2_test.csv",dtype={"ID":str})
champ=pd.read_csv(DATA_DIR/"submission_v14.csv",dtype={"ID":str})

assert len(train)==3297 and len(valid)==500 and len(test)==1000
assert test.ID.tolist()==champ.ID.tolist()
assert test.ID.tolist()==v2t.ID.tolist()
assert valid.ID.tolist()==v2v.ID.tolist()
assert test.ID.head(4).tolist()==[
    "BICH_0003","BICH_0006","BICH_0015","BICH_0016"
]

print("✅ LIVE Kaggle partition verified")
print("train/validation/test:",len(train),len(valid),len(test))
print("first test IDs:",test.ID.head(6).tolist())

print("Prevalence:")
display(pd.DataFrame({
    "train":train[LABELS].mean(),
    "validation":valid[LABELS].mean()
}).round(5))

def clean_text(x):
    x=unicodedata.normalize("NFKC",str(x))
    x=x.replace("\u200c"," ").replace("\u200d"," ")
    x=re.sub(r"\s+"," ",x).strip()
    try:
        x=normalize(x)
    except Exception:
        pass
    return re.sub(r"\s+"," ",x).strip()

def raw_clean(x):
    x=unicodedata.normalize("NFKC",str(x))
    return re.sub(r"\s+"," ",x).strip()

def group_key(x):
    return raw_clean(x).lower()

for df in [train,valid,test]:
    df["model_text"]=df.text.map(clean_text)
    df["raw_text"]=df.text.map(raw_clean)
    df["group_key"]=df.text.map(group_key)

combined=pd.concat([train,valid],ignore_index=True)


In [ ]:

# 4. LIVE combined grouped folds — used only for new OOF models
groups=combined.groupby("group_key",sort=False).agg(
    size=("ID","size"),
    **{f"sum_{l}":(l,"sum") for l in LABELS}
).reset_index()

global_pos=np.array([combined[l].sum() for l in LABELS],float)
total=float(len(combined))
rarity=1/np.sqrt(np.maximum(global_pos,1))
mat=groups[[f"sum_{l}" for l in LABELS]].to_numpy(float)

rng=np.random.default_rng(SEED)
priority=(mat*rarity).sum(1)+0.005*groups["size"].to_numpy()
order=np.argsort(-(priority+rng.random(len(groups))*1e-8))

fp=np.zeros((5,len(LABELS)))
fs=np.zeros(5)
assign={}

for idx in order:
    row=groups.iloc[idx]
    vec=np.array([row[f"sum_{l}"] for l in LABELS],float)
    size=float(row["size"])
    costs=[]

    for f in range(5):
        p=fp.copy(); s=fs.copy()
        p[f]+=vec; s[f]+=size
        costs.append(
            np.mean((p/np.maximum(global_pos[None,:],1)-.2)**2)
            +.30*np.mean((s/total-.2)**2)
        )

    best=int(np.argmin(costs))
    assign[row.group_key]=best
    fp[best]+=vec
    fs[best]+=size

combined["fold"]=combined.group_key.map(assign).astype(int)

assert combined.groupby("group_key").fold.nunique().max()==1
assert set(combined.fold)==set(range(5))

print(combined.groupby("fold").size().to_dict())
display(combined.groupby("fold")[LABELS].sum())


In [ ]:

# 5. Dataset and attention-pooling model
class DS(Dataset):
    def __init__(self,df,tok,labels=True):
        self.x=df.model_text.astype(str).tolist()
        self.y=df[LABELS].to_numpy(np.float32) if labels else None
        self.tok=tok

    def __len__(self):
        return len(self.x)

    def __getitem__(self,i):
        e=self.tok(
            self.x[i],truncation=True,max_length=MAX_LENGTH,padding=False
        )
        if self.y is not None:
            e["labels"]=self.y[i]
        return e

class Coll:
    def __init__(self,tok):
        self.tok=tok

    def __call__(self,items):
        y=None
        if "labels" in items[0]:
            y=torch.tensor(
                np.stack([x.pop("labels") for x in items]),
                dtype=torch.float32
            )

        b=self.tok.pad(
            items,padding=True,pad_to_multiple_of=8,return_tensors="pt"
        )
        if y is not None:
            b["labels"]=y
        return b

class AttentionPooling(nn.Module):
    def __init__(self,h):
        super().__init__()
        self.scorer=nn.Linear(h,1)

    def forward(self,hs,mask):
        s=self.scorer(hs).squeeze(-1)
        s=s.masked_fill(mask==0,-1e4)
        a=torch.softmax(s,dim=1).unsqueeze(-1)
        return (hs*a).sum(1)

class Stage2Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.base=AutoModel.from_pretrained(MODEL_NAME)
        h=self.base.config.hidden_size
        self.pool=AttentionPooling(h)
        self.head=nn.Sequential(
            nn.Dropout(.1),
            nn.Linear(h*2,512),
            nn.GELU(),
            nn.Dropout(.2),
            nn.Linear(512,7)
        )

    def forward(self,input_ids,attention_mask,token_type_ids=None):
        kw={"input_ids":input_ids,"attention_mask":attention_mask}
        if token_type_ids is not None:
            kw["token_type_ids"]=token_type_ids

        out=self.base(**kw)
        hs=out.last_hidden_state
        cls=hs[:,0]
        att=self.pool(hs,attention_mask)

        return self.head(torch.cat([cls,att],dim=1))

@torch.no_grad()
def predict(model,loader,custom=False):
    model.eval()
    parts=[]

    for b in loader:
        b.pop("labels",None)
        inp={k:v.to("cuda",non_blocking=True) for k,v in b.items()}

        with torch.autocast(device_type="cuda",dtype=torch.float16):
            logits=model(**inp) if custom else model(**inp).logits

        parts.append(torch.sigmoid(logits.float()).cpu().numpy())

    return np.concatenate(parts)


In [ ]:

# 6. Stage 1 binary toxicity BanglaBERT — LIVE combined 5-fold OOF
fold_seeds=[2026,17,3407,1337,42]

def train_stage1():
    tok=AutoTokenizer.from_pretrained(MODEL_NAME)
    oof=np.zeros(len(combined),np.float32)
    test_parts=[]
    hist=[]

    for fold in range(5):
        tr=combined[combined.fold!=fold].copy()
        va=combined[combined.fold==fold].copy()

        cache=RUN_DIR/f"stage1_fold_{fold}.npz"
        hp=RUN_DIR/f"stage1_fold_{fold}_history.csv"

        if cache.exists():
            with np.load(cache) as z:
                idx=z["idx"]; pv=z["valid"]; pt=z["test"]
            if np.array_equal(idx,va.index.to_numpy()):
                print("✅ reuse stage1",fold)
                oof[idx]=pv
                test_parts.append(pt)
                continue

        seed_all(fold_seeds[fold])
        gc.collect()
        torch.cuda.empty_cache()

        col=Coll(tok)
        tl=DataLoader(
            DS(tr,tok,True),batch_size=BATCH,shuffle=True,
            num_workers=0,pin_memory=True,collate_fn=col
        )
        vl=DataLoader(
            DS(va,tok,True),batch_size=EVAL_BATCH,shuffle=False,
            num_workers=0,pin_memory=True,collate_fn=col
        )
        tel=DataLoader(
            DS(test,tok,False),batch_size=EVAL_BATCH,shuffle=False,
            num_workers=0,pin_memory=True,collate_fn=col
        )

        model=AutoModelForSequenceClassification.from_pretrained(
            MODEL_NAME,num_labels=1
        ).to("cuda")

        opt=torch.optim.AdamW(
            model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY
        )

        steps=math.ceil(len(tl)/ACCUM)*EPOCHS
        sched=get_linear_schedule_with_warmup(
            opt,int(steps*WARMUP_RATIO),steps
        )
        scaler=torch.amp.GradScaler("cuda")

        sv=[]; st=[]; fh=[]

        for epoch in range(1,EPOCHS+1):
            model.train()
            opt.zero_grad(set_to_none=True)
            run=0.; seen=0

            for step,b in enumerate(
                tqdm(tl,desc=f"stage1 f{fold} e{epoch}/{EPOCHS}")
            ):
                y=b.pop("labels")[:,0].to("cuda",non_blocking=True)
                inp={k:v.to("cuda",non_blocking=True) for k,v in b.items()}

                with torch.autocast(device_type="cuda",dtype=torch.float16):
                    logits=model(**inp).logits[:,0]

                loss=torch.nn.functional.binary_cross_entropy_with_logits(
                    logits.float(),y
                )
                scaler.scale(loss/ACCUM).backward()

                run+=float(loss.item())*len(y)
                seen+=len(y)

                if (step+1)%ACCUM==0 or step+1==len(tl):
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
                    old=scaler.get_scale()
                    scaler.step(opt)
                    scaler.update()
                    if scaler.get_scale()>=old:
                        sched.step()
                    opt.zero_grad(set_to_none=True)

            rec={"fold":fold,"epoch":epoch,"loss":run/max(seen,1)}
            print(rec)
            fh.append(rec); hist.append(rec)
            pd.DataFrame(fh).to_csv(hp,index=False)

            if epoch in SNAPSHOT_EPOCHS:
                sv.append(predict(model,vl)[:,0])
                st.append(predict(model,tel)[:,0])

        pv=np.mean(sv,axis=0)
        pt=np.mean(st,axis=0)
        oof[va.index.to_numpy()]=pv
        test_parts.append(pt)

        np.savez_compressed(
            cache,idx=va.index.to_numpy(),valid=pv,test=pt
        )

        del model,opt,sched,scaler,tl,vl,tel
        gc.collect()
        torch.cuda.empty_cache()

    ptest=np.mean(test_parts,axis=0)

    np.savez_compressed(
        RUN_DIR/"stage1_oof_test.npz",oof=oof,test=ptest
    )
    pd.DataFrame(hist).to_csv(
        RUN_DIR/"stage1_training_history.csv",index=False
    )

    return oof,ptest

s1_oof,s1_test=train_stage1()
print("✅ Stage 1 complete")


In [ ]:

# 7. Stage 2 hard-negative attention model
# Select globally difficult non-toxic rows using Stage-1 OOF rank.
non_idx=np.where(combined.is_toxic.to_numpy()==0)[0]
k=max(1,int(round(.40*len(non_idx))))
hard_non=np.zeros(len(combined),bool)
hard_non[non_idx[np.argsort(-s1_oof[non_idx])[:k]]]=True

print("Hard non-toxic selected:",int(hard_non.sum()))

def stage2_loss(logits,y,frame):
    raw=torch.nn.functional.binary_cross_entropy_with_logits(
        logits.float(),y,reduction="none"
    )

    pos=frame[LABELS].sum().to_numpy(float)
    neg=len(frame)-pos
    boost=(np.maximum(neg,1)/np.maximum(pos,1))**.18
    boost=np.clip(boost,1.,1.8)
    boost[0]=1.

    pb=torch.tensor(boost,dtype=torch.float32,device=logits.device).unsqueeze(0)
    w=torch.where(y>.5,pb,torch.ones_like(y))

    task=torch.tensor(
        [.50]+[.50/6]*6,dtype=torch.float32,device=logits.device
    ).unsqueeze(0)

    return (raw*w*task).sum(1).mean()

def train_stage2():
    tok=AutoTokenizer.from_pretrained(MODEL_NAME)
    oof=np.zeros((len(combined),7),np.float32)
    test_parts=[]
    hist=[]

    for fold in range(5):
        train_mask=combined.fold.to_numpy()!=fold
        use=train_mask & (
            (combined.is_toxic.to_numpy()==1) | hard_non
        )

        tr=combined[use].copy()
        va=combined[combined.fold==fold].copy()

        cache=RUN_DIR/f"stage2_fold_{fold}.npz"
        hp=RUN_DIR/f"stage2_fold_{fold}_history.csv"

        if cache.exists():
            with np.load(cache) as z:
                idx=z["idx"]; pv=z["valid"]; pt=z["test"]
            if np.array_equal(idx,va.index.to_numpy()):
                print("✅ reuse stage2",fold)
                oof[idx]=pv
                test_parts.append(pt)
                continue

        print(
            f"stage2 fold {fold}: train={len(tr)} "
            f"toxic={int(tr.is_toxic.sum())} "
            f"hard_non={int((tr.is_toxic==0).sum())}"
        )

        seed_all(fold_seeds[fold]+100)
        gc.collect()
        torch.cuda.empty_cache()

        col=Coll(tok)
        tl=DataLoader(
            DS(tr,tok,True),batch_size=BATCH,shuffle=True,
            num_workers=0,pin_memory=True,collate_fn=col
        )
        vl=DataLoader(
            DS(va,tok,True),batch_size=EVAL_BATCH,shuffle=False,
            num_workers=0,pin_memory=True,collate_fn=col
        )
        tel=DataLoader(
            DS(test,tok,False),batch_size=EVAL_BATCH,shuffle=False,
            num_workers=0,pin_memory=True,collate_fn=col
        )

        model=Stage2Model().to("cuda")

        backbone=[]; head=[]
        for n,p in model.named_parameters():
            (backbone if n.startswith("base.") else head).append(p)

        opt=torch.optim.AdamW([
            {"params":backbone,"lr":LR,"weight_decay":WEIGHT_DECAY},
            {"params":head,"lr":HEAD_LR,"weight_decay":WEIGHT_DECAY},
        ])

        steps=math.ceil(len(tl)/ACCUM)*EPOCHS
        sched=get_linear_schedule_with_warmup(
            opt,int(steps*WARMUP_RATIO),steps
        )
        scaler=torch.amp.GradScaler("cuda")

        sv=[]; st=[]; fh=[]

        for epoch in range(1,EPOCHS+1):
            model.train()
            opt.zero_grad(set_to_none=True)
            run=0.; seen=0

            for step,b in enumerate(
                tqdm(tl,desc=f"stage2 f{fold} e{epoch}/{EPOCHS}")
            ):
                y=b.pop("labels").to("cuda",non_blocking=True)
                inp={k:v.to("cuda",non_blocking=True) for k,v in b.items()}

                with torch.autocast(device_type="cuda",dtype=torch.float16):
                    logits=model(**inp)

                loss=stage2_loss(logits,y,tr)
                scaler.scale(loss/ACCUM).backward()

                run+=float(loss.item())*len(y)
                seen+=len(y)

                if (step+1)%ACCUM==0 or step+1==len(tl):
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
                    old=scaler.get_scale()
                    scaler.step(opt)
                    scaler.update()
                    if scaler.get_scale()>=old:
                        sched.step()
                    opt.zero_grad(set_to_none=True)

            rec={"fold":fold,"epoch":epoch,"loss":run/max(seen,1)}
            print(rec)
            fh.append(rec); hist.append(rec)
            pd.DataFrame(fh).to_csv(hp,index=False)

            if epoch in SNAPSHOT_EPOCHS:
                sv.append(predict(model,vl,custom=True))
                st.append(predict(model,tel,custom=True))

        pv=np.mean(sv,axis=0)
        pt=np.mean(st,axis=0)

        oof[va.index.to_numpy()]=pv
        test_parts.append(pt)

        np.savez_compressed(
            cache,idx=va.index.to_numpy(),valid=pv,test=pt
        )

        del model,opt,sched,scaler,tl,vl,tel
        gc.collect()
        torch.cuda.empty_cache()

    ptest=np.mean(test_parts,axis=0)

    np.savez_compressed(
        RUN_DIR/"stage2_oof_test.npz",oof=oof,test=ptest
    )
    pd.DataFrame(hist).to_csv(
        RUN_DIR/"stage2_training_history.csv",index=False
    )

    return oof,ptest

s2_oof,s2_test=train_stage2()
print("✅ Stage 2 complete")


In [ ]:

# 8. Cheap lexical OOF expert on LIVE combined data
lex_oof=np.zeros((len(combined),7),np.float32)
lex_test_parts=[]

for fold in range(5):
    tr=combined[combined.fold!=fold]
    va=combined[combined.fold==fold]

    cv=TfidfVectorizer(
        analyzer="char_wb",ngram_range=(2,6),min_df=2,
        max_features=160000,sublinear_tf=True,dtype=np.float32
    )
    wv=TfidfVectorizer(
        analyzer="word",ngram_range=(1,2),min_df=2,
        max_features=60000,sublinear_tf=True,dtype=np.float32,
        token_pattern=r"(?u)\b\w+\b"
    )

    Ctr=cv.fit_transform(tr.raw_text)
    Cva=cv.transform(va.raw_text)
    Cte=cv.transform(test.raw_text)

    Wtr=wv.fit_transform(tr.raw_text)
    Wva=wv.transform(va.raw_text)
    Wte=wv.transform(test.raw_text)

    Xtr=hstack([Ctr,Wtr],format="csr")
    Xva=hstack([Cva,Wva],format="csr")
    Xte=hstack([Cte,Wte],format="csr")

    pv=np.zeros((len(va),7),np.float32)
    pt=np.zeros((len(test),7),np.float32)

    for j,l in enumerate(LABELS):
        clf=LogisticRegression(
            C=2.5,solver="liblinear",max_iter=1000,random_state=SEED+fold
        )
        clf.fit(Xtr,tr[l].to_numpy(int))
        pv[:,j]=clf.predict_proba(Xva)[:,1]
        pt[:,j]=clf.predict_proba(Xte)[:,1]

    lex_oof[va.index.to_numpy()]=pv
    lex_test_parts.append(pt)
    print("lex fold",fold,Xtr.shape)

lex_test=np.mean(lex_test_parts,axis=0)

np.savez_compressed(
    RUN_DIR/"lexical_oof_test.npz",oof=lex_oof,test=lex_test
)
print("✅ Lexical OOF complete")


In [ ]:

# 9. Load all reusable LIVE OOF sources
def load(name):
    return np.load(DATA_DIR/name)

v15=load("v15_specialist.npz")
v17b=load("v17_bangla_toxic.npz")
v17x=load("v17_xlmr_toxic.npz")
v18=load("v18_muril.npz")
v19=load("v19_mdeberta_toxic.npz")
v23=load("v23_conditional.npz")
v25=load("v25_banglabert_large_toxic.npz")

oldb=load("old_bangla_snapshots.npz")
oldx=load("old_xlmr_snapshots.npz")

# Shape sanity
assert v18["oof"].shape==(3797,7)
assert v23["oof"].shape==(3797,6)
assert v15["oof"].shape==(3797,4)
assert s1_oof.shape==(3797,)
assert s2_oof.shape==(3797,7)
assert lex_oof.shape==(3797,7)

print("✅ All LIVE OOF sources aligned")


In [ ]:

# 10. Reconstruct V14 validation probability family
def logit(p):
    p=np.clip(np.asarray(p,float),1e-6,1-1e-6)
    return np.log(p/(1-p))

def sigmoid(z):
    return 1/(1+np.exp(-z))

v14v={}
v14t={}

v14v["is_toxic"]=(
    .875*v2v.is_toxic_prob.to_numpy(float)
    +.125*oldx["valid"][1:3,:,0].mean(0)
)
v14t["is_toxic"]=(
    .875*v2t.is_toxic_prob.to_numpy(float)
    +.125*oldx["test"][1:3,:,0].mean(0)
)

j=LI["implicit_toxicity"]
v14v["implicit_toxicity"]=sigmoid(
    .9*logit(v2v.implicit_toxicity_prob)
    +.1*logit(oldb["valid"].mean(0)[:,j])
)
v14t["implicit_toxicity"]=sigmoid(
    .9*logit(v2t.implicit_toxicity_prob)
    +.1*logit(oldb["test"].mean(0)[:,j])
)

v14v["sarcasm"]=v2v.sarcasm_prob.to_numpy(float)
v14t["sarcasm"]=v2t.sarcasm_prob.to_numpy(float)

j=LI["contains_profanity"]
v14v["contains_profanity"]=(
    .95*v2v.contains_profanity_prob.to_numpy(float)
    +.05*oldb["valid"][0,:,j]
)
v14t["contains_profanity"]=(
    .95*v2t.contains_profanity_prob.to_numpy(float)
    +.05*oldb["test"][0,:,j]
)

j=LI["threatening"]
v14v["threatening"]=(
    .95*v2v.threatening_prob.to_numpy(float)
    +.05*oldx["valid"][0,:,j]
)
v14t["threatening"]=(
    .95*v2t.threatening_prob.to_numpy(float)
    +.05*oldx["test"][0,:,j]
)

j=LI["identity_attack"]
v14v["identity_attack"]=(
    .70*v2v.identity_attack_prob.to_numpy(float)
    +.30*oldb["valid"][2,:,j]
)
v14t["identity_attack"]=(
    .70*v2t.identity_attack_prob.to_numpy(float)
    +.30*oldb["test"][2,:,j]
)

v14v["sexual_content"]=v2v.sexual_content_prob.to_numpy(float)
v14t["sexual_content"]=v2t.sexual_content_prob.to_numpy(float)

print("✅ V14 probability family ready")


In [ ]:

# 11. Build CLEAN train-only meta features
# First 3297 OOF rows are live train; last 500 are live validation.

NTR=len(train)
VAL_IDX=np.arange(NTR,NTR+len(valid))

# v15 column map
V15_LABELS=[
    "implicit_toxicity","contains_profanity","threatening","sexual_content"
]
V15I={l:i for i,l in enumerate(V15_LABELS)}
V23I={l:i for i,l in enumerate(ATTRS)}

def features_for_label(label,split):
    if split=="train":
        sl=slice(0,NTR)
        s1=s1_oof[sl]
        s2=s2_oof[sl]
        lex=lex_oof[sl]
    elif split=="validation":
        sl=VAL_IDX
        s1=s1_oof[sl]
        s2=s2_oof[sl]
        lex=lex_oof[sl]
    else:
        s1=s1_test
        s2=s2_test
        lex=lex_test

    j=LI[label]
    cols=[s2[:,j],lex[:,j]]
    names=["stage2","lexical"]

    if label=="is_toxic":
        cols.append(s1); names.append("stage1")

        sources=[
            ("v17_bangla",v17b),
            ("v17_xlmr",v17x),
            ("v19_mdeberta",v19),
            ("v25_bangla_large",v25),
        ]

        for name,d in sources:
            arr=d["oof"][sl] if split!="test" else d["test"]
            cols.append(arr); names.append(name)

        arr=v18["oof"][sl,0] if split!="test" else v18["test"][:,0]
        cols.append(arr); names.append("muril")

    else:
        arr=v18["oof"][sl,j] if split!="test" else v18["test"][:,j]
        cols.append(arr); names.append("muril")

        # Conditional specialist
        c=v23["oof"][sl,V23I[label]] if split!="test" else v23["test"][:,V23I[label]]
        cols.append(c); names.append("conditional")

        if label in V15I:
            c=v15["oof"][sl,V15I[label]] if split!="test" else v15["test"][:,V15I[label]]
            cols.append(c); names.append("v15_specialist")

    raw=np.column_stack(cols)
    lg=logit(raw)

    return np.column_stack([raw,lg]),names

print({
    l:features_for_label(l,"train")[0].shape
    for l in LABELS
})


In [ ]:

# 12. Regularized meta-stackers trained on 3297 TRAIN OOF only
ytr=train[LABELS].to_numpy(int)
yva=valid[LABELS].to_numpy(int)

meta_val=np.zeros((len(valid),7),float)
meta_test=np.zeros((len(test),7),float)
meta_report=[]

for j,label in enumerate(LABELS):
    X,names=features_for_label(label,"train")
    Xv,_=features_for_label(label,"validation")
    Xt,_=features_for_label(label,"test")

    pipe=Pipeline([
        ("scale",StandardScaler()),
        ("lr",LogisticRegression(
            C=.08,
            solver="liblinear",
            max_iter=1000,
            random_state=SEED
        ))
    ])

    pipe.fit(X,ytr[:,j])

    meta_val[:,j]=pipe.predict_proba(Xv)[:,1]
    meta_test[:,j]=pipe.predict_proba(Xt)[:,1]

    meta_report.append({
        "label":label,
        "base_sources":"|".join(names),
        "features":X.shape[1]
    })

pd.DataFrame(meta_report).to_csv(
    RUN_DIR/"meta_report.csv",index=False
)

print("✅ Train-only meta stackers built")


In [ ]:

# 13. Exact metric + rank calibration
def f1_pos(y,p):
    y=np.asarray(y,int); p=np.asarray(p,int)
    tp=((y==1)&(p==1)).sum()
    fp=((y==0)&(p==1)).sum()
    fn=((y==1)&(p==0)).sum()
    d=2*tp+fp+fn
    return float(2*tp/d) if d else 0.

def f1_macro(y,p):
    return float(np.mean([
        f1_pos((y==c).astype(int),(p==c).astype(int))
        for c in [0,1]
    ]))

def metric(label,y,p):
    return f1_macro(y,p) if label=="is_toxic" else f1_pos(y,p)

def official(y,pred):
    ss=[metric(l,y[:,j],pred[:,j]) for j,l in enumerate(LABELS)]
    return float(.5*ss[0]+.5*np.mean(ss[1:])),ss

def topk(prob,rate):
    k=int(round(rate*len(prob)))
    k=max(0,min(k,len(prob)))
    out=np.zeros(len(prob),int)
    if k:
        out[np.argsort(-prob)[:k]]=1
    return out

def best_topk(label,y,p):
    order=np.argsort(-p)
    pred=np.zeros(len(p),int)

    best=(metric(label,y,pred),0)

    for k in range(1,len(p)+1):
        pred[order[k-1]]=1
        s=metric(label,y,pred)
        if s>best[0]:
            best=(s,k)

    return best


In [ ]:

# 14. Final controlled validation selection
# Candidate families: V14, meta, Stage2, lexical.
# For each source we compare exact validation-optimal rank and 3 prior-regularized rates.

val_pred=np.zeros((len(valid),7),int)
test_pred=np.zeros((len(test),7),int)
rows=[]
selected={}

combined_prior=pd.concat(
    [train[LABELS],valid[LABELS]],ignore_index=True
).mean()

for j,label in enumerate(LABELS):
    candidates={
        "v14":(v14v[label],v14t[label]),
        "meta":(meta_val[:,j],meta_test[:,j]),
        "stage2":(s2_oof[VAL_IDX,j],s2_test[:,j]),
        "lexical":(lex_oof[VAL_IDX,j],lex_test[:,j]),
    }

    if label=="is_toxic":
        candidates["stage1"]=(s1_oof[VAL_IDX],s1_test)

    best=None

    for name,(pv,pt) in candidates.items():
        # A) exact validation-rank optimum
        sc,k=best_topk(label,yva[:,j],pv)
        rate=k/len(valid)

        # Apply same rank percentage to test.
        vp=topk(pv,rate)
        tp=topk(pt,rate)

        gap=abs(rate-float(yva[:,j].mean()))
        robust=sc-.05*gap

        rec={
            "label":label,"source":name,"policy":"validation_topk",
            "validation_score":sc,"rate":rate,
            "validation_true_rate":float(yva[:,j].mean()),
            "robust":robust
        }
        rows.append(rec)

        key=(robust,sc,-gap)

        if best is None or key>best["key"]:
            best={"key":key,"rec":rec,"vp":vp,"tp":tp}

        # B) fixed prevalence neighborhood — much safer for hidden shift.
        for scale in [.90,1.00,1.10]:
            r=float(train[label].mean())*scale
            vp=topk(pv,r)
            tp=topk(pt,float(combined_prior[label])*scale)
            sc=metric(label,yva[:,j],vp)
            gap=abs(float(vp.mean())-float(yva[:,j].mean()))
            robust=sc-.05*gap

            rec={
                "label":label,"source":name,"policy":"prior_topk",
                "scale":scale,"validation_score":sc,
                "rate":float(vp.mean()),
                "validation_true_rate":float(yva[:,j].mean()),
                "robust":robust
            }
            rows.append(rec)

            key=(robust,sc,-gap)
            if best is None or key>best["key"]:
                best={"key":key,"rec":rec,"vp":vp,"tp":tp}

    selected[label]=best
    val_pred[:,j]=best["vp"]
    test_pred[:,j]=best["tp"]

selection=pd.DataFrame(rows)
selection.to_csv(
    RUN_DIR/"v32_validation_selection.csv",index=False
)

score,label_scores=official(yva,val_pred)

report=pd.DataFrame({
    "label":LABELS,
    "source":[selected[l]["rec"]["source"] for l in LABELS],
    "policy":[selected[l]["rec"]["policy"] for l in LABELS],
    "validation_score":label_scores,
    "validation_true_rate":yva.mean(0),
    "validation_pred_rate":val_pred.mean(0),
    "test_pred_rate":test_pred.mean(0),
})
report.to_csv(
    RUN_DIR/"v32_final_validation_report.csv",index=False
)

print("V32 LIVE validation official score:",score)
display(report.round(5))


In [ ]:

# 15. Build ONLY correct-LIVE-ID submission
sub=champ.copy()

for j,l in enumerate(LABELS):
    sub[l]=test_pred[:,j].astype(int)

assert sub.ID.tolist()==test.ID.tolist()
assert sub.ID.tolist()==champ.ID.tolist()
assert sub.ID.head(4).tolist()==[
    "BICH_0003","BICH_0006","BICH_0015","BICH_0016"
]
assert sub[LABELS].isin([0,1]).all().all()

SUBMIT_CANDIDATE=bool(score>=.600)

summary={
    "version":"v32_LIVE_cascade_oof_superensemble",
    "live_public_champion":0.586,
    "validation_official_score":float(score),
    "attribute_mean":float(np.mean(label_scores[1:])),
    "toxic_macro_f1":float(label_scores[0]),
    "SUBMIT_CANDIDATE":SUBMIT_CANDIDATE,
    "live_test_first_ids":sub.ID.head(6).tolist(),
    "warning":"Send v32_results.zip to ChatGPT before Kaggle submission."
}

(RUN_DIR/"v32_summary.json").write_text(
    json.dumps(summary,indent=2)
)

sub.to_csv(
    RUN_DIR/"submission_v32_LIVE.csv",index=False
)

print(json.dumps(summary,indent=2))

if SUBMIT_CANDIDATE:
    print("\n✅ V32 passed the 0.600 LIVE validation gate.")
else:
    print("\n❌ V32 did not pass 0.600. Do not submit yet.")


In [ ]:

# 16. Package results
np.savez_compressed(
    RUN_DIR/"v32_new_probabilities.npz",
    stage1_oof=s1_oof,stage1_test=s1_test,
    stage2_oof=s2_oof,stage2_test=s2_test,
    lexical_oof=lex_oof,lexical_test=lex_test,
    meta_validation=meta_val,meta_test=meta_test
)

zip_path=Path("/content/v32_results.zip")

with zipfile.ZipFile(
    zip_path,"w",compression=zipfile.ZIP_DEFLATED
) as z:
    for p in RUN_DIR.iterdir():
        if p.is_file():
            z.write(p,arcname=p.name)

print("Created:",zip_path)

from google.colab import files
files.download(str(zip_path))
